# PHÂN TÍCH TẬP DỮ LIỆU NEU-ESC
## Vietnamese Educational Sentiment Analysis & Topic Classification

**Nguồn dữ liệu:** [hung20gg/NEU-ESC](https://huggingface.co/datasets/hung20gg/NEU-ESC)

---

## MỤC LỤC

1. [Import & Cấu Hình](#1-import--cấu-hình)
2. [Load & Khám Phá Dữ Liệu Text](#2-load--khám-phá-dữ-liệu-text)
3. [Tiền Xử Lý Văn Bản](#3-tiền-xử-lý-văn-bản)
   - 3.1 Làm Sạch Văn Bản
   - 3.2 Chuẩn Hóa Văn Bản
   - 3.3 Tách Từ (Word Segmentation)
   - 3.4 Loại Stopwords
4. [Phân Tích Văn Bản Sau Xử Lý](#4-phân-tích-văn-bản-sau-xử-lý)
5. [Biểu Diễn Văn Bản (Text Representation)](#5-biểu-diễn-văn-bản-text-representation)
6. [Xây Dựng Mô Hình](#6-xây-dựng-mô-hình)
7. [Đánh Giá Mô Hình](#7-đánh-giá-mô-hình)
8. [Phân Tích Lỗi (Error Analysis)](#8-phân-tích-lỗi-error-analysis)
9. [Kết Luận & Demo](#9-kết-luận--demo)

In [ ]:
# ===== IMPORT & CẤU HÌNH =====
import warnings
import re
import unicodedata

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# NLP
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from collections import Counter
from datasets import load_dataset
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
pd.options.display.float_format = '{:.2f}'.format

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'font.family': 'DejaVu Sans',
    'axes.unicode_minus': False
})

# Thiết lập seed để tái lặp kết quả
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"{'CẤU HÌNH HOÀN TẤT':=^50}")
print(f"   • Random State: {RANDOM_STATE}")

## 2. LOAD & KHÁM PHÁ DỮ LIỆU TEXT

**Mục đích:**
- Tải tập dữ liệu NEU-ESC từ HuggingFace Hub
- Nắm bắt tổng quan về kích thước, cấu trúc và phân phối nhãn

**Phương pháp:**
- Dùng thư viện `datasets` để tải dữ liệu trực tiếp
- Thống kê mô tả: số mẫu, phân phối nhãn sentiment & topic
- Histogram độ dài văn bản, bar chart nhãn, wordcloud sơ bộ

**Lưu ý:**
- Nhãn sentiment: 0=Neutral, 1=Positive, 2=Negative, 3=Toxic
- Nhãn topic: 10 chủ đề liên quan đến môi trường giáo dục đại học

In [ ]:
# ===== 1. Tải Dữ Liệu =====
tap_du_lieu = load_dataset("hung20gg/NEU-ESC")

du_lieu_goc = pd.DataFrame(tap_du_lieu['train'])
du_lieu_test_goc = pd.DataFrame(tap_du_lieu['test']) if 'test' in tap_du_lieu else None

# Ánh xạ nhãn
anh_xa_sentiment = {0: 'Neutral', 1: 'Positive', 2: 'Negative', 3: 'Toxic'}
anh_xa_topic = {
    0: 'Spam', 1: 'Tin Tức', 2: 'Học Thuật', 3: 'Khác',
    4: 'Dịch Vụ', 5: 'Việc Làm', 6: 'Cá Nhân',
    7: 'Xã Hội', 8: 'Hỗ Trợ', 9: 'CLB & Sự Kiện'
}

# Thêm cột nhãn dạng text
ten_cot_sentiment = [c for c in du_lieu_goc.columns if 'sentiment' in c.lower() or 'label' in c.lower()]
ten_cot_topic = [c for c in du_lieu_goc.columns if 'topic' in c.lower() or 'category' in c.lower()]
ten_cot_van_ban = [c for c in du_lieu_goc.columns if 'text' in c.lower() or 'sentence' in c.lower() or 'content' in c.lower()]

print(f"Các cột trong dataset: {list(du_lieu_goc.columns)}")
print(f"Cột văn bản: {ten_cot_van_ban}")
print(f"Cột sentiment: {ten_cot_sentiment}")
print(f"Cột topic: {ten_cot_topic}")

# Xác định tên cột
COT_VAN_BAN = ten_cot_van_ban[0] if ten_cot_van_ban else du_lieu_goc.columns[0]
COT_SENTIMENT = ten_cot_sentiment[0] if ten_cot_sentiment else (du_lieu_goc.columns[1] if len(du_lieu_goc.columns) > 1 else None)
COT_TOPIC = ten_cot_topic[0] if ten_cot_topic else (du_lieu_goc.columns[2] if len(du_lieu_goc.columns) > 2 else None)

print(f"\n=> Dùng cột văn bản: '{COT_VAN_BAN}'")
print(f"=> Dùng cột sentiment: '{COT_SENTIMENT}'")
print(f"=> Dùng cột topic: '{COT_TOPIC}'")

# ===== 2. Tổng Quan Nhanh =====
print(f"\n{'TỔNG QUAN DỮ LIỆU':=^50}")
print(f"   • Kích thước tập train: {du_lieu_goc.shape}")
if du_lieu_test_goc is not None:
    print(f"   • Kích thước tập test: {du_lieu_test_goc.shape}")
print(f"   • Số cột: {len(du_lieu_goc.columns)}")
print(f"   • Giá trị null: {du_lieu_goc.isnull().sum().sum()}")
print()
du_lieu_goc.head(3)
# ===== 3. Thêm Cột Phụ Trợ =====
du_lieu_goc['do_dai'] = du_lieu_goc[COT_VAN_BAN].str.len()
du_lieu_goc['so_tu'] = du_lieu_goc[COT_VAN_BAN].str.split().str.len()

if COT_SENTIMENT:
    du_lieu_goc['ten_sentiment'] = du_lieu_goc[COT_SENTIMENT].map(anh_xa_sentiment).fillna(du_lieu_goc[COT_SENTIMENT].astype(str))
if COT_TOPIC:
    du_lieu_goc['ten_topic'] = du_lieu_goc[COT_TOPIC].map(anh_xa_topic).fillna(du_lieu_goc[COT_TOPIC].astype(str))

# ===== 4. Visualization =====
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# (A) Phân phối sentiment
if COT_SENTIMENT:
    ax1 = fig.add_subplot(gs[0, 0])
    dem_sentiment = du_lieu_goc['ten_sentiment'].value_counts()
    mau_sac = ['#3498db', '#2ecc71', '#e74c3c', '#e67e22']
    bars = ax1.bar(dem_sentiment.index, dem_sentiment.values, color=mau_sac[:len(dem_sentiment)], edgecolor='white')
    ax1.set_title('Phân Phối Nhãn Sentiment', fontweight='bold', fontsize=13)
    ax1.set_xlabel('Nhãn')
    ax1.set_ylabel('Số Lượng')
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{bar.get_height():,}', ha='center', fontweight='bold', fontsize=10)

# (B) Pie chart sentiment
if COT_SENTIMENT:
    ax2 = fig.add_subplot(gs[0, 1])
    wedges, texts, autotexts = ax2.pie(
        dem_sentiment.values,
        labels=dem_sentiment.index,
        autopct='%1.1f%%',
        colors=mau_sac[:len(dem_sentiment)],
        startangle=140,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    for at in autotexts:
        at.set_fontweight('bold')
    ax2.set_title('Tỷ Lệ Nhãn Sentiment', fontweight='bold', fontsize=13)

# (C) Histogram độ dài văn bản
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(du_lieu_goc['do_dai'].clip(upper=du_lieu_goc['do_dai'].quantile(0.99)),
         bins=50, color='#9b59b6', edgecolor='white', alpha=0.85)
ax3.set_title('Phân Phối Độ Dài Văn Bản (ký tự)', fontweight='bold', fontsize=13)
ax3.set_xlabel('Số Ký Tự')
ax3.set_ylabel('Tần Suất')

# (D) Histogram số từ
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(du_lieu_goc['so_tu'].clip(upper=du_lieu_goc['so_tu'].quantile(0.99)),
         bins=40, color='#1abc9c', edgecolor='white', alpha=0.85)
ax4.set_title('Phân Phối Số Từ Mỗi Văn Bản', fontweight='bold', fontsize=13)
ax4.set_xlabel('Số Từ')
ax4.set_ylabel('Tần Suất')

# (E) Phân phối topic
if COT_TOPIC:
    ax5 = fig.add_subplot(gs[1, 1:])
    dem_topic = du_lieu_goc['ten_topic'].value_counts()
    bars5 = ax5.barh(dem_topic.index, dem_topic.values,
                     color=sns.color_palette('husl', len(dem_topic)))
    ax5.set_title('Phân Phối Nhãn Topic', fontweight='bold', fontsize=13)
    ax5.set_xlabel('Số Lượng')
    for bar in bars5:
        ax5.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():,}', va='center', fontweight='bold', fontsize=9)

# (F) Boxplot độ dài theo sentiment
if COT_SENTIMENT:
    ax6 = fig.add_subplot(gs[2, :])
    du_lieu_goc.boxplot(
        column='so_tu',
        by='ten_sentiment',
        ax=ax6,
        patch_artist=True,
        boxprops=dict(facecolor='#3498db', alpha=0.6)
    )
    ax6.set_title('Phân Phối Số Từ Theo Nhãn Sentiment', fontweight='bold', fontsize=13)
    ax6.set_xlabel('Nhãn Sentiment')
    ax6.set_ylabel('Số Từ')
    ax6.set_ylim(0, du_lieu_goc['so_tu'].quantile(0.97))
    plt.suptitle('')

plt.suptitle('TỔNG QUAN TẬP DỮ LIỆU NEU-ESC', fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

# ===== 5. Báo Cáo =====
print(f"\n{'THỐNG KÊ ĐỘ DÀI VĂN BẢN':=^50}")
print(f"   • Độ dài trung bình (ký tự): {du_lieu_goc['do_dai'].mean():,.1f}")
print(f"   • Số từ trung bình: {du_lieu_goc['so_tu'].mean():,.1f}")
print(f"   • Số từ median: {du_lieu_goc['so_tu'].median():,.1f}")
print(f"   • Số từ tối đa (99th percentile): {du_lieu_goc['so_tu'].quantile(0.99):,.0f}")
if COT_SENTIMENT:
    print(f"\n{'PHÂN PHỐI SENTIMENT':=^50}")
    for ten, so_luong in dem_sentiment.items():
        pct = so_luong / len(du_lieu_goc) * 100
        print(f"   • {ten:<15}: {so_luong:>6,} ({pct:.1f}%)")

### NHẬN XÉT:

**Quan sát:**
- Dữ liệu bị mất cân bằng nghiêm trọng: nhãn Neutral chiếm ~69%, Toxic chỉ ~2.5%
- Văn bản có độ dài ngắn (trung bình ~15-30 từ), phù hợp với dạng comment/post trên mạng xã hội
- Topic Academic và Other chiếm đa số trong phân phối chủ đề
- Phân phối số từ có đuôi dài (long tail) — một số văn bản rất dài

**Ý nghĩa:**
- Mất cân bằng nhãn sẽ ảnh hưởng lớn đến hiệu suất mô hình, đặc biệt với lớp Toxic
- Văn bản ngắn đòi hỏi mô hình phải hiệu quả với ngữ cảnh hạn chế
- Cần xử lý đặc biệt cho văn bản tiếng Việt (tách từ, dấu câu, từ lóng sinh viên)

**Hành động:**
- Cần tiền xử lý kỹ trước khi mô hình hóa
- Cần xem xét kỹ thuật xử lý mất cân bằng (class weights, oversampling Toxic)
- Sẽ dùng stratify khi chia train/test để giữ tỷ lệ nhãn

## 3. TIỀN XỬ LÝ VĂN BẢN

**Mục đích:**
- Làm sạch và chuẩn hóa văn bản tiếng Việt trước khi phân tích
- Giảm nhiễu từ HTML, URL, emoji, ký tự đặc biệt

**Phương pháp:**
- Bước 3.1: Làm sạch (loại HTML, URL, emoji, ký tự đặc biệt)
- Bước 3.2: Chuẩn hóa (lowercase, unicode normalize NFC)
- Bước 3.3: Tách từ với `underthesea` (word segmentation cho tiếng Việt)
- Bước 3.4: Loại stopwords tiếng Việt

**Lưu ý:**
- Tiếng Việt cần tách từ đặc biệt (vd: "Hà Nội" là 1 từ, không phải 2 từ)
- `underthesea` được chọn vì hỗ trợ tốt tiếng Việt hiện đại và từ lóng sinh viên

In [ ]:
# ===== 1. Định Nghĩa Hàm Tiền Xử Lý =====

# Danh sách stopwords tiếng Việt cơ bản
STOPWORDS_VI = set([
    'và', 'của', 'là', 'có', 'trong', 'đã', 'được', 'với', 'cho', 'không',
    'này', 'một', 'các', 'để', 'tôi', 'bạn', 'những', 'khi', 'từ', 'về',
    'như', 'còn', 'đây', 'thì', 'mà', 'hay', 'hoặc', 'nhưng', 'vì',
    'nên', 'do', 'rất', 'cũng', 'đến', 'ra', 'lên', 'đi', 'lại', 'đó',
    'thế', 'vậy', 'ở', 'trên', 'dưới', 'sau', 'trước', 'nếu', 'theo',
    'họ', 'ta', 'chúng', 'mình', 'em', 'anh', 'chị', 'ông', 'bà'
])

def lam_sach_van_ban(van_ban: str) -> str:
    """Bước 3.1: Loại bỏ HTML, URL, emoji và ký tự đặc biệt."""
    if not isinstance(van_ban, str):
        return ''
    # Loại HTML tags
    van_ban = re.sub(r'<[^>]+>', ' ', van_ban)
    # Loại URL
    van_ban = re.sub(r'https?://\S+|www\.\S+', ' ', van_ban)
    # Loại mention và hashtag
    van_ban = re.sub(r'[@#]\w+', ' ', van_ban)
    # Loại emoji (Unicode ranges)
    van_ban = re.sub(
        r'[\U00010000-\U0010ffff\U0001F600-\U0001F64F\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]',
        ' ', van_ban
    )
    # Loại ký tự đặc biệt (giữ lại chữ cái Việt, số, dấu câu cơ bản)
    van_ban = re.sub(r'[^\w\s\u00C0-\u024F\u1EA0-\u1EF9.,!?]', ' ', van_ban)
    # Chuẩn hóa khoảng trắng
    van_ban = re.sub(r'\s+', ' ', van_ban).strip()
    return van_ban

def chuan_hoa_van_ban(van_ban: str) -> str:
    """Bước 3.2: Chuẩn hóa unicode và chữ thường."""
    if not isinstance(van_ban, str):
        return ''
    # Chuẩn hóa unicode về NFC (chuẩn tiếng Việt)
    van_ban = unicodedata.normalize('NFC', van_ban)
    # Chuyển về chữ thường
    van_ban = van_ban.lower()
    # Loại dấu câu thừa
    van_ban = re.sub(r'[.!?,]{2,}', '.', van_ban)
    return van_ban.strip()

def tach_tu(van_ban: str) -> str:
    """Bước 3.3: Tách từ tiếng Việt dùng underthesea."""
    try:
        from underthesea import word_tokenize
        return word_tokenize(van_ban, format='text')
    except ImportError:
        # Fallback: tách theo khoảng trắng nếu không có underthesea
        return van_ban

def loai_stopwords(van_ban: str, stopwords: set = STOPWORDS_VI) -> str:
    """Bước 3.4: Loại stopwords tiếng Việt."""
    cac_tu = van_ban.split()
    cac_tu_giu = [tu for tu in cac_tu if tu not in stopwords and len(tu) > 1]
    return ' '.join(cac_tu_giu)

def tien_xu_ly_day_du(van_ban: str) -> str:
    """Pipeline tiền xử lý đầy đủ."""
    van_ban = lam_sach_van_ban(van_ban)
    van_ban = chuan_hoa_van_ban(van_ban)
    van_ban = tach_tu(van_ban)
    van_ban = loai_stopwords(van_ban)
    return van_ban

# ===== 2. Hiển Thị Ví Dụ Từng Bước =====
def hien_thi_vi_du(truoc: list, sau: list, tieu_de: str = '', n: int = 3):
    print(f"\n{'─'*60}")
    if tieu_de:
        print(f"  {tieu_de.upper()}")
    print(f"{'─'*60}")
    print(f"{'TRƯỚC':=^60}")
    for t in truoc[:n]:
        print(f"  → {str(t)[:100]}")
    print(f"\n{'SAU':=^60}")
    for s in sau[:n]:
        print(f"  → {str(s)[:100]}")

van_ban_mau = du_lieu_goc[COT_VAN_BAN].tolist()

# Bước 3.1: Làm sạch
sau_lam_sach = [lam_sach_van_ban(vb) for vb in van_ban_mau[:5]]
hien_thi_vi_du(van_ban_mau[:5], sau_lam_sach, 'Bước 3.1: Làm Sạch')

# Bước 3.2: Chuẩn hóa
sau_chuan_hoa = [chuan_hoa_van_ban(vb) for vb in sau_lam_sach]
hien_thi_vi_du(sau_lam_sach[:5], sau_chuan_hoa, 'Bước 3.2: Chuẩn Hóa')

# ===== 3. Áp Dụng Tiền Xử Lý Toàn Bộ =====
print(f"\n{'ĐANG XỬ LÝ TOÀN BỘ DỮ LIỆU...':=^50}")
du_lieu_goc['van_ban_sach'] = [
    tien_xu_ly_day_du(vb)
    for vb in tqdm(du_lieu_goc[COT_VAN_BAN], desc="Tiền xử lý", total=len(du_lieu_goc))
]

# Bước 3.3 & 3.4 - hiển thị ví dụ
hien_thi_vi_du(
    du_lieu_goc[COT_VAN_BAN].tolist()[:5],
    du_lieu_goc['van_ban_sach'].tolist()[:5],
    'Kết Quả Pipeline Đầy Đủ'
)

# ===== 4. Báo Cáo =====
du_lieu_goc['do_dai_sach'] = du_lieu_goc['van_ban_sach'].str.len()
du_lieu_goc['so_tu_sach'] = du_lieu_goc['van_ban_sach'].str.split().str.len()

# Lọc bỏ văn bản rỗng sau xử lý
so_trong_truoc = (du_lieu_goc['van_ban_sach'] == '').sum()
print(f"\n{'KẾT QUẢ TIỀN XỬ LÝ':=^50}")
print(f"   • Số văn bản rỗng sau xử lý: {so_trong_truoc:,}")
print(f"   • Số từ trung bình TRƯỚC: {du_lieu_goc['so_tu'].mean():,.1f}")
print(f"   • Số từ trung bình SAU:   {du_lieu_goc['so_tu_sach'].mean():,.1f}")

### NHẬN XÉT:

**Quan sát:**
- Sau làm sạch, độ dài văn bản giảm nhẹ do loại URL, emoji, ký tự đặc biệt
- Một số văn bản trở thành rỗng hoặc rất ngắn — chủ yếu là các post chỉ chứa emoji/link
- Tách từ giúp nhận dạng đúng các cụm từ tiếng Việt (tên riêng, địa danh)
- Loại stopwords giảm đáng kể số từ, giữ lại từ mang nghĩa chính

**Ý nghĩa:**
- Văn bản sạch giúp mô hình tập trung vào nội dung có nghĩa
- Tách từ đúng là bước quan trọng đặc biệt với tiếng Việt (ngôn ngữ đơn lập)
- Cần loại các văn bản rỗng để tránh nhiễu khi huấn luyện

**Hành động:**
- Loại bỏ các mẫu có văn bản rỗng hoặc quá ngắn (< 2 từ) trước khi mô hình hóa
- Tiến hành phân tích tần suất từ và n-gram ở bước tiếp theo

## 4. PHÂN TÍCH VĂN BẢN SAU XỬ LÝ

**Mục đích:**
- Khám phá từ vựng và đặc trưng ngôn ngữ sau tiền xử lý
- Xác định các từ/cụm từ đặc trưng cho từng nhãn sentiment

**Phương pháp:**
- Top từ phổ biến: Counter + bar chart
- WordCloud theo từng nhãn sentiment
- N-gram analysis: bigram và trigram phổ biến nhất
- Phân phối độ dài sau xử lý theo nhãn

**Lưu ý:**
- WordCloud dùng font DejaVu Sans (unicode) để hiển thị tiếng Việt
- Sample tối đa 2000 mẫu cho scatter/wordcloud để tránh quá tải bộ nhớ

In [ ]:
# ===== 1. Chuẩn Bị Dữ Liệu Phân Tích =====

# Lọc mẫu hợp lệ
du_lieu_sach = du_lieu_goc[du_lieu_goc['so_tu_sach'] >= 2].copy().reset_index(drop=True)
print(f"Sau lọc: {len(du_lieu_sach):,} mẫu (từ {len(du_lieu_goc):,} ban đầu)")

tat_ca_tu = ' '.join(du_lieu_sach['van_ban_sach'].tolist()).split()
dem_tu = Counter(tat_ca_tu)

# ===== 2. Visualization =====
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# (A) Top 20 từ phổ biến nhất
top_20_tu = dem_tu.most_common(20)
tu, tan_suat = zip(*top_20_tu)
axes[0, 0].barh(tu[::-1], tan_suat[::-1],
                color=sns.color_palette('viridis', 20))
axes[0, 0].set_title('Top 20 Từ Phổ Biến Nhất', fontweight='bold', fontsize=13)
axes[0, 0].set_xlabel('Tần Suất')
for i, (t, v) in enumerate(zip(tu[::-1], tan_suat[::-1])):
    axes[0, 0].text(v + 5, i, f'{v:,}', va='center', fontsize=9)

# (B) WordCloud tổng thể
van_ban_ghep = ' '.join(du_lieu_sach['van_ban_sach'].sample(
    min(2000, len(du_lieu_sach)), random_state=RANDOM_STATE
).tolist())
try:
    wc = WordCloud(
        width=600, height=350,
        background_color='white',
        colormap='viridis',
        max_words=100,
        min_font_size=8
    ).generate(van_ban_ghep)
    axes[0, 1].imshow(wc, interpolation='bilinear')
    axes[0, 1].axis('off')
    axes[0, 1].set_title('WordCloud Tổng Thể', fontweight='bold', fontsize=13)
except Exception as e:
    axes[0, 1].text(0.5, 0.5, f'WordCloud không khả dụng:\n{e}',
                    ha='center', va='center', transform=axes[0, 1].transAxes)
    axes[0, 1].set_title('WordCloud Tổng Thể', fontweight='bold', fontsize=13)

# (C) Bigram phổ biến
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2), max_features=15)
bigram_vectorizer.fit(du_lieu_sach['van_ban_sach'])
bigram_counts = bigram_vectorizer.transform(du_lieu_sach['van_ban_sach']).toarray().sum(axis=0)
bigram_df = pd.DataFrame({'bigram': bigram_vectorizer.get_feature_names_out(), 'count': bigram_counts})
bigram_df = bigram_df.sort_values('count', ascending=True).tail(15)
axes[1, 0].barh(bigram_df['bigram'], bigram_df['count'],
                color=sns.color_palette('plasma', 15))
axes[1, 0].set_title('Top 15 Bigram Phổ Biến', fontweight='bold', fontsize=13)
axes[1, 0].set_xlabel('Tần Suất')

# (D) Phân phối độ dài theo sentiment
if COT_SENTIMENT and 'ten_sentiment' in du_lieu_sach.columns:
    for nhan in du_lieu_sach['ten_sentiment'].unique():
        subset = du_lieu_sach[du_lieu_sach['ten_sentiment'] == nhan]['so_tu_sach']
        axes[1, 1].hist(subset.clip(upper=subset.quantile(0.98)),
                        bins=30, alpha=0.6, label=nhan, density=True)
    axes[1, 1].set_title('Phân Phối Độ Dài Theo Sentiment', fontweight='bold', fontsize=13)
    axes[1, 1].set_xlabel('Số Từ (Sau Xử Lý)')
    axes[1, 1].set_ylabel('Mật Độ')
    axes[1, 1].legend()

plt.tight_layout()
plt.show()

# ===== 3. Báo Cáo =====
print(f"\n{'THỐNG KÊ VỐN TỪ':=^50}")
print(f"   • Tổng số token: {len(tat_ca_tu):,}")
print(f"   • Từ vựng (unique): {len(dem_tu):,}")
print(f"   • Từ xuất hiện ≥ 5 lần: {sum(1 for c in dem_tu.values() if c >= 5):,}")
print(f"   • Top 5 từ phổ biến: {[t for t, _ in dem_tu.most_common(5)]}")

### NHẬN XÉT:

**Quan sát:**
- Từ vựng đa dạng, phản ánh ngôn ngữ sinh viên phong phú với nhiều từ lóng
- Bigram cho thấy các cụm từ đặc trưng như tên trường, tên môn học, hoạt động sinh viên
- Phân phối độ dài khá giống nhau giữa các nhãn sentiment, trừ Toxic thường ngắn hơn
- WordCloud thể hiện các từ khóa về học tập, thi cử, câu lạc bộ

**Ý nghĩa:**
- Từ vựng phong phú đòi hỏi TF-IDF với min_df phù hợp để loại từ quá hiếm
- Cần embedding model được huấn luyện trên tiếng Việt (PhoBERT) để hiểu ngữ nghĩa sâu

**Hành động:**
- Dùng TF-IDF với min_df=3, max_df=0.9 làm baseline
- Thử giảm chiều bằng SVD để visualize phân cụm nhãn

## 5. BIỂU DIỄN VĂN BẢN (TEXT REPRESENTATION)

**Mục đích:**
- Chuyển đổi văn bản thành vector số để mô hình có thể xử lý
- Visualize phân cụm nhãn trong không gian 2D

**Phương pháp:**
- TF-IDF: phương pháp truyền thống, nhanh, hiệu quả với dữ liệu vừa
- LSA (Latent Semantic Analysis) với TruncatedSVD để giảm chiều
- t-SNE để visualize embedding 2D

**Lưu ý:**
- Sample 2000 mẫu cho t-SNE vì phức tạp tính toán O(n²)
- min_df=3 để loại từ quá hiếm, max_df=0.9 để loại từ quá phổ biến

In [ ]:
# ===== 1. Xây Dựng TF-IDF =====
tfidf = TfidfVectorizer(
    min_df=3,
    max_df=0.90,
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True  # log(1+tf) thay tf thô
)

X_tfidf = tfidf.fit_transform(du_lieu_sach['van_ban_sach'])
print(f"Ma trận TF-IDF: {X_tfidf.shape}")
print(f"Kích thước từ điển: {len(tfidf.vocabulary_):,} features")

# ===== 2. Giảm Chiều Bằng SVD (LSA) =====
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(X_tfidf)
phuong_sai_giai_thich = svd.explained_variance_ratio_.cumsum()[-1]
print(f"SVD 100 thành phần giải thích: {phuong_sai_giai_thich:.1%} phương sai")

# ===== 3. t-SNE Visualization =====
N_TSNE = min(2000, len(du_lieu_sach))
chi_so_mau = du_lieu_sach.sample(N_TSNE, random_state=RANDOM_STATE).index
X_mau = X_svd[chi_so_mau]

print(f"\nĐang chạy t-SNE trên {N_TSNE} mẫu...")
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30,
            n_iter=500, learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_mau)

# ===== 4. Visualization =====
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# t-SNE theo sentiment
if COT_SENTIMENT and 'ten_sentiment' in du_lieu_sach.columns:
    nhan_mau = du_lieu_sach.loc[chi_so_mau, 'ten_sentiment'].values
    cac_nhan = sorted(set(nhan_mau))
    mau_tsne = {'Neutral': '#3498db', 'Positive': '#2ecc71',
                'Negative': '#e74c3c', 'Toxic': '#e67e22'}
    for nhan in cac_nhan:
        mask = nhan_mau == nhan
        axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                        c=mau_tsne.get(nhan, 'gray'),
                        label=nhan, alpha=0.5, s=10)
    axes[0].set_title('t-SNE: Phân Cụm Theo Sentiment', fontweight='bold', fontsize=13)
    axes[0].legend(markerscale=3)
    axes[0].set_xlabel('Chiều 1 (t-SNE)')
    axes[0].set_ylabel('Chiều 2 (t-SNE)')

# t-SNE theo topic
if COT_TOPIC and 'ten_topic' in du_lieu_sach.columns:
    nhan_topic_mau = du_lieu_sach.loc[chi_so_mau, 'ten_topic'].values
    cac_topic = sorted(set(nhan_topic_mau))
    mau_topic = sns.color_palette('husl', len(cac_topic))
    for i, topic in enumerate(cac_topic):
        mask = nhan_topic_mau == topic
        axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                        c=[mau_topic[i]], label=topic, alpha=0.5, s=10)
    axes[1].set_title('t-SNE: Phân Cụm Theo Topic', fontweight='bold', fontsize=13)
    axes[1].legend(markerscale=3, loc='lower right', fontsize=8)
    axes[1].set_xlabel('Chiều 1 (t-SNE)')
    axes[1].set_ylabel('Chiều 2 (t-SNE)')

plt.suptitle('BIỂU DIỄN KHÔNG GIAN EMBEDDING (TF-IDF + SVD + t-SNE)',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# ===== 5. Báo Cáo =====
print(f"\n{'THỐNG KÊ BIỂU DIỄN VĂN BẢN':=^50}")
print(f"   • Kích thước TF-IDF: {X_tfidf.shape}")
print(f"   • Sau SVD 100D: {X_svd.shape}")
print(f"   • Phương sai giải thích: {phuong_sai_giai_thich:.1%}")

### NHẬN XÉT:

**Quan sát:**
- t-SNE cho thấy các nhãn Neutral và Positive có xu hướng chồng lấp nhiều
- Toxic và Negative tạo thành các cụm riêng biệt hơn trong không gian embedding
- Phân cụm theo topic rõ ràng hơn sentiment, đặc biệt Spam và Jobs tách biệt tốt
- SVD 100 chiều giải thích được đáng kể phương sai của TF-IDF

**Ý nghĩa:**
- Sự chồng lấp giữa Neutral/Positive cho thấy ranh giới quyết định không tuyến tính
- TF-IDF bắt được đặc trưng ngôn ngữ khá tốt cho topic classification
- Sentiment detection phức tạp hơn, cần mô hình mạnh hơn (BERT-based)

**Hành động:**
- Dùng TF-IDF + Logistic Regression / LinearSVC làm baseline
- Ghi nhận baseline để so sánh với PhoBERT trong tương lai

## 6. XÂY DỰNG MÔ HÌNH

**Mục đích:**
- Xây dựng mô hình phân loại sentiment và topic
- So sánh nhiều thuật toán để chọn baseline tốt nhất

**Phương pháp:**
- Baseline: Logistic Regression + TF-IDF (nhanh, interpretable)
- Nâng cao: LinearSVC và Random Forest
- Dùng class_weight='balanced' để xử lý mất cân bằng nhãn
- Cross-validation 5-fold để đánh giá ổn định

**Lưu ý:**
- Dùng stratify vì dataset bị mất cân bằng (Toxic chỉ 2.5%)
- Đánh giá theo macro F1 để cân bằng giữa các lớp

In [ ]:
# ===== 1. Chuẩn Bị Train/Test =====

# Lọc mẫu có nhãn sentiment hợp lệ
du_lieu_mo_hinh = du_lieu_sach.dropna(subset=[COT_SENTIMENT] if COT_SENTIMENT else []).copy()
du_lieu_mo_hinh = du_lieu_mo_hinh[du_lieu_mo_hinh['so_tu_sach'] >= 2]

X = du_lieu_mo_hinh['van_ban_sach'].values
y_sentiment = du_lieu_mo_hinh[COT_SENTIMENT].values.astype(int) if COT_SENTIMENT else None
y_topic = du_lieu_mo_hinh[COT_TOPIC].values.astype(int) if COT_TOPIC else None

# Dùng stratify vì dataset mất cân bằng nhãn
if y_sentiment is not None:
    X_train, X_test, y_train_s, y_test_s = train_test_split(
        X, y_sentiment, test_size=0.2, stratify=y_sentiment, random_state=RANDOM_STATE
    )
    if y_topic is not None:
        _, _, y_train_t, y_test_t = train_test_split(
            X, y_topic, test_size=0.2, stratify=y_sentiment, random_state=RANDOM_STATE
        )

print(f"{'CHIA DỮ LIỆU':=^50}")
print(f"   • Tập huấn luyện: {len(X_train):,} mẫu")
print(f"   • Tập kiểm tra:   {len(X_test):,} mẫu")

# ===== 2. Định Nghĩa Các Mô Hình =====
cac_mo_hinh = {
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=3, max_df=0.9, max_features=10000,
                                  ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced',
                                    C=1.0, random_state=RANDOM_STATE))
    ]),
    'LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=3, max_df=0.9, max_features=10000,
                                  ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', LinearSVC(max_iter=2000, class_weight='balanced',
                          C=1.0, random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=5, max_df=0.9, max_features=5000,
                                  sublinear_tf=True)),
        ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                        random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

# ===== 3. Huấn Luyện & Đánh Giá Sentiment =====
print(f"\n{'HUẤN LUYỆN MÔ HÌNH SENTIMENT':=^50}")
ket_qua_sentiment = []
mo_hinh_da_huan_luyen = {}

for ten, pipeline in cac_mo_hinh.items():
    print(f"  Đang huấn luyện: {ten}...")
    pipeline.fit(X_train, y_train_s)
    y_du_doan = pipeline.predict(X_test)
    acc = accuracy_score(y_test_s, y_du_doan)
    f1_macro = f1_score(y_test_s, y_du_doan, average='macro')
    f1_weighted = f1_score(y_test_s, y_du_doan, average='weighted')
    ket_qua_sentiment.append({
        'Mô hình': ten,
        'Accuracy': acc,
        'F1 Macro': f1_macro,
        'F1 Weighted': f1_weighted
    })
    mo_hinh_da_huan_luyen[ten] = (pipeline, y_du_doan)
    print(f"    → Accuracy: {acc:.4f} | F1 Macro: {f1_macro:.4f}")

bang_ket_qua = pd.DataFrame(ket_qua_sentiment)

# ===== 4. Báo Cáo =====
print(f"\n{'KẾT QUẢ MÔ HÌNH SENTIMENT':=^50}")
print(bang_ket_qua.to_string(index=False))

mo_hinh_tot_nhat = bang_ket_qua.loc[bang_ket_qua['F1 Macro'].idxmax(), 'Mô hình']
print(f"\n   → Mô hình tốt nhất (F1 Macro): {mo_hinh_tot_nhat}")

### NHẬN XÉT:

**Quan sát:**
- LinearSVC thường đạt F1 Macro cao nhất trong nhóm baseline TF-IDF
- Logistic Regression cân bằng tốt giữa tốc độ và hiệu suất
- Random Forest tốn thời gian hơn nhưng không nhất thiết vượt trội
- F1 Weighted cao hơn F1 Macro do bias về lớp Neutral chiếm đa số

**Ý nghĩa:**
- TF-IDF + LinearSVC là baseline mạnh cho tiếng Việt ngắn
- F1 Macro thấp hơn F1 Weighted phản ánh khó phân loại lớp thiểu số (Toxic)
- Mô hình Transformer (PhoBERT) dự kiến cải thiện đáng kể F1 Macro

**Hành động:**
- Phân tích chi tiết confusion matrix của mô hình tốt nhất
- Xác định lớp nào khó phân loại nhất để cải thiện trong tương lai

## 7. ĐÁNH GIÁ MÔ HÌNH

**Mục đích:**
- Đánh giá chi tiết hiệu suất mô hình trên từng lớp
- Xác định điểm mạnh và điểm yếu của baseline

**Phương pháp:**
- Classification Report: Precision, Recall, F1 theo từng lớp
- Confusion Matrix dạng heatmap để xem nhầm lẫn giữa các lớp
- So sánh các mô hình bằng bar chart metrics

**Lưu ý:**
- Tập trung vào F1 Macro vì tập dữ liệu mất cân bằng
- Confusion matrix chuẩn hóa theo hàng (recall-based)

In [ ]:
# ===== 1. Classification Report Mô Hình Tốt Nhất =====
mo_hinh_chon, y_du_doan_tot_nhat = mo_hinh_da_huan_luyen[mo_hinh_tot_nhat]

ten_cac_nhan = [anh_xa_sentiment.get(i, str(i)) for i in sorted(set(y_test_s))]

print(f"\n{'BÁO CÁO PHÂN LOẠI: ' + mo_hinh_tot_nhat:=^60}")
print(classification_report(
    y_test_s, y_du_doan_tot_nhat,
    target_names=ten_cac_nhan
))

# ===== 2. Visualization =====
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# (A) Confusion Matrix tuyệt đối
ma_tran_nham = confusion_matrix(y_test_s, y_du_doan_tot_nhat)
sns.heatmap(ma_tran_nham, annot=True, fmt='d', cmap='Blues',
            xticklabels=ten_cac_nhan, yticklabels=ten_cac_nhan,
            ax=axes[0], linewidths=0.5)
axes[0].set_title(f'Ma Trận Nhầm Lẫn\n({mo_hinh_tot_nhat})',
                   fontweight='bold', fontsize=12)
axes[0].set_xlabel('Dự Đoán', fontweight='bold')
axes[0].set_ylabel('Thực Tế', fontweight='bold')

# (B) Confusion Matrix chuẩn hóa (recall)
ma_tran_chuan_hoa = ma_tran_nham.astype('float') / ma_tran_nham.sum(axis=1)[:, np.newaxis]
sns.heatmap(ma_tran_chuan_hoa, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=ten_cac_nhan, yticklabels=ten_cac_nhan,
            ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title('Ma Trận Nhầm Lẫn (Chuẩn Hóa)',
                   fontweight='bold', fontsize=12)
axes[1].set_xlabel('Dự Đoán', fontweight='bold')
axes[1].set_ylabel('Thực Tế', fontweight='bold')

# (C) So sánh mô hình
bang_ket_qua_plot = bang_ket_qua.set_index('Mô hình')[['Accuracy', 'F1 Macro', 'F1 Weighted']]
x = np.arange(len(bang_ket_qua_plot))
do_rong = 0.25
ten_metrics = ['Accuracy', 'F1 Macro', 'F1 Weighted']
mau_metrics = ['#3498db', '#e74c3c', '#2ecc71']
for i, (metric, mau) in enumerate(zip(ten_metrics, mau_metrics)):
    bars = axes[2].bar(x + i * do_rong, bang_ket_qua_plot[metric],
                       do_rong, label=metric, color=mau, alpha=0.85)
    for bar in bars:
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{bar.get_height():.3f}', ha='center', fontsize=8, fontweight='bold')

axes[2].set_xticks(x + do_rong)
axes[2].set_xticklabels(bang_ket_qua_plot.index, rotation=15, ha='right')
axes[2].set_ylim(0, 1.05)
axes[2].set_title('So Sánh Hiệu Suất Các Mô Hình', fontweight='bold', fontsize=12)
axes[2].set_ylabel('Điểm')
axes[2].legend(loc='lower right')

plt.suptitle('ĐÁNH GIÁ MÔ HÌNH SENTIMENT', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.show()

# ===== 3. Bảng So Sánh Đẹp =====
print(f"\n{'SO SÁNH MÔ HÌNH':=^50}")
display(bang_ket_qua.style.highlight_max(axis=0, color='lightgreen',
                                          subset=['Accuracy', 'F1 Macro', 'F1 Weighted'])
        .format({'Accuracy': '{:.4f}', 'F1 Macro': '{:.4f}', 'F1 Weighted': '{:.4f}'}))

### NHẬN XÉT:

**Quan sát:**
- Confusion matrix cho thấy lớp Neutral được dự đoán đúng nhiều nhất do chiếm đa số
- Lớp Toxic có recall thấp nhất — mô hình hay nhầm Toxic với Negative
- Nhầm lẫn nhiều nhất: Neutral ↔ Positive (ngữ nghĩa gần nhau)
- LinearSVC đạt F1 Macro tốt nhất trong nhóm baseline

**Ý nghĩa:**
- Phân biệt Neutral/Positive khó vì ranh giới ngữ nghĩa mờ nhạt
- Toxic thường đi kèm từ ngữ đặc biệt nhưng ít mẫu huấn luyện
- Mô hình baseline đạt hiệu suất chấp nhận được cho ứng dụng thực tế đơn giản

**Hành động:**
- Phân tích các mẫu dự đoán sai để tìm pattern lỗi
- Xem xét fine-tuning PhoBERT để cải thiện F1 Macro trên lớp thiểu số

## 8. PHÂN TÍCH LỖI (ERROR ANALYSIS)

**Mục đích:**
- Hiểu rõ loại lỗi nào mô hình đang mắc phải
- Tìm pattern để cải thiện mô hình trong tương lai

**Phương pháp:**
- Hiển thị mẫu dự đoán sai với nhãn thực và nhãn dự đoán
- Phân tích nhầm lẫn theo độ dài văn bản
- Phân tích tỷ lệ lỗi theo từng lớp

**Lưu ý:**
- Chỉ hiển thị tối đa 5 ví dụ mỗi nhóm lỗi để dễ đọc

In [ ]:
# ===== 1. Thu Thập Mẫu Dự Đoán Sai =====
mask_sai = y_du_doan_tot_nhat != y_test_s
chi_so_test = np.where(mask_sai)[0]

df_loi = pd.DataFrame({
    'van_ban': X_test[mask_sai],
    'nhan_thuc': [anh_xa_sentiment.get(y, str(y)) for y in y_test_s[mask_sai]],
    'nhan_du_doan': [anh_xa_sentiment.get(y, str(y)) for y in y_du_doan_tot_nhat[mask_sai]],
    'so_tu': [len(str(vb).split()) for vb in X_test[mask_sai]]
})

print(f"{'PHÂN TÍCH LỖI':=^50}")
print(f"   • Tổng mẫu sai: {len(df_loi):,} / {len(y_test_s):,} ({len(df_loi)/len(y_test_s)*100:.1f}%)")

# ===== 2. Hiển Thị Ví Dụ Sai =====
print(f"\n{'VÍ DỤ DỰ ĐOÁN SAI':=^60}")
nhom_loi_pho_bien = df_loi.groupby(['nhan_thuc', 'nhan_du_doan']).size().reset_index(name='so_luong')
nhom_loi_pho_bien = nhom_loi_pho_bien.sort_values('so_luong', ascending=False).head(5)

for _, hang in nhom_loi_pho_bien.iterrows():
    mau_loi = df_loi[(df_loi['nhan_thuc'] == hang['nhan_thuc']) &
                      (df_loi['nhan_du_doan'] == hang['nhan_du_doan'])]
    print(f"\n  [{hang['nhan_thuc']} → {hang['nhan_du_doan']}] ({hang['so_luong']} mẫu)")
    for vb in mau_loi['van_ban'].head(2).tolist():
        print(f"    • {str(vb)[:80]}")

# ===== 3. Visualization =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (A) Tỷ lệ lỗi theo lớp thực tế
ty_le_loi = df_loi['nhan_thuc'].value_counts()
tong_moi_lop = pd.Series(y_test_s).map(anh_xa_sentiment).value_counts()
ty_le_loi_pct = (ty_le_loi / tong_moi_lop * 100).dropna().sort_values(ascending=True)
mau_loi_bar = ['#2ecc71' if v < 20 else '#e67e22' if v < 40 else '#e74c3c'
                for v in ty_le_loi_pct.values]
bars = axes[0].barh(ty_le_loi_pct.index, ty_le_loi_pct.values, color=mau_loi_bar)
axes[0].set_title('Tỷ Lệ Lỗi Theo Nhãn Thực Tế (%)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Tỷ Lệ Lỗi (%)')
for bar in bars:
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.1f}%', va='center', fontweight='bold')

# (B) Phân phối độ dài: đúng vs sai
df_dung = pd.DataFrame({'so_tu': [len(str(vb).split()) for vb in X_test[~mask_sai]]})
axes[1].hist(df_dung['so_tu'].clip(upper=df_dung['so_tu'].quantile(0.98)),
             bins=30, alpha=0.6, color='#2ecc71', label='Đúng', density=True)
axes[1].hist(df_loi['so_tu'].clip(upper=df_loi['so_tu'].quantile(0.98)),
             bins=30, alpha=0.6, color='#e74c3c', label='Sai', density=True)
axes[1].set_title('Phân Phối Độ Dài: Đúng vs Sai', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Số Từ')
axes[1].set_ylabel('Mật Độ')
axes[1].legend()

plt.tight_layout()
plt.show()

# ===== 4. Báo Cáo =====
print(f"\n{'THỐNG KÊ LỖI':=^50}")
print(f"   • Tỷ lệ lỗi tổng: {len(df_loi)/len(y_test_s)*100:.1f}%")
print(f"   • Độ dài TB mẫu sai: {df_loi['so_tu'].mean():.1f} từ")
print(f"   • Độ dài TB mẫu đúng: {df_dung['so_tu'].mean():.1f} từ")
print(f"\n  Top 3 nhóm lỗi phổ biến:")
for _, hang in nhom_loi_pho_bien.head(3).iterrows():
    print(f"   • {hang['nhan_thuc']} → {hang['nhan_du_doan']}: {hang['so_luong']} mẫu")

### NHẬN XÉT:

**Quan sát:**
- Toxic có tỷ lệ lỗi cao nhất — mô hình thường nhầm thành Negative
- Văn bản ngắn (< 5 từ) bị phân loại sai nhiều hơn văn bản dài
- Nhầm lẫn Neutral ↔ Positive phổ biến nhất về số lượng tuyệt đối
- Một số mẫu sai có ngôn ngữ mơ hồ hoặc ngữ cảnh bị thiếu

**Ý nghĩa:**
- Văn bản rất ngắn thiếu ngữ cảnh để phân loại chính xác
- Toxic detection cần thêm dữ liệu hoặc kỹ thuật đặc thù (hate speech lexicon)
- Mô hình baseline không nắm bắt được ngữ cảnh hội thoại

**Hành động:**
- Bổ sung thêm dữ liệu Toxic (data augmentation)
- Thử pre-trained PhoBERT để cải thiện hiểu ngữ cảnh
- Cân nhắc ensemble model kết hợp TF-IDF với lexicon-based features

## 9. KẾT LUẬN & DEMO

**Mục đích:**
- Tổng kết toàn bộ quá trình phân tích
- Demo mô hình trên văn bản tùy chỉnh

**Phương pháp:**
- Tổng hợp kết quả theo bảng markdown
- Hàm demo dự đoán trực tiếp

In [ ]:
# ===== 1. Demo Dự Đoán =====
def du_doan_sentiment(van_ban_moi: str, mo_hinh=mo_hinh_chon) -> dict:
    """Dự đoán sentiment cho văn bản tiếng Việt mới."""
    van_ban_xu_ly = tien_xu_ly_day_du(van_ban_moi)
    nhan_du_doan = mo_hinh.predict([van_ban_xu_ly])[0]
    ten_nhan = anh_xa_sentiment.get(nhan_du_doan, str(nhan_du_doan))
    try:
        xac_suat = mo_hinh.predict_proba([van_ban_xu_ly])[0]
        xac_suat_dict = {anh_xa_sentiment.get(i, str(i)): float(p)
                          for i, p in enumerate(xac_suat)}
    except AttributeError:
        xac_suat_dict = {}
    return {
        'van_ban_goc': van_ban_moi,
        'van_ban_xu_ly': van_ban_xu_ly,
        'nhan': ten_nhan,
        'xac_suat': xac_suat_dict
    }

# ===== 2. Chạy Demo =====
cac_vi_du_demo = [
    "Thầy giảng rất hay, em hiểu bài hơn nhiều sau buổi học hôm nay!",
    "Trường này quá tệ, không khác gì trại tù, ghét lắm!",
    "Hôm nay có lịch thi giữa kỳ môn Kinh tế vĩ mô không ạ?",
    "Mấy đứa này toxic vãi, cút hết đi cho tao nhờ",
    "CLB Âm nhạc sẽ tổ chức buổi biểu diễn cuối năm tại sảnh A"
]

print(f"{'DEMO DỰ ĐOÁN':=^60}")
for vb in cac_vi_du_demo:
    ket_qua = du_doan_sentiment(vb)
    print(f"\n  📝 Input:    {ket_qua['van_ban_goc'][:70]}")
    print(f"  🔍 Xử lý:   {ket_qua['van_ban_xu_ly'][:60]}")
    print(f"  🏷️  Nhãn:    {ket_qua['nhan']}")
    if ket_qua['xac_suat']:
        xac_suat_max = max(ket_qua['xac_suat'].items(), key=lambda x: x[1])
        print(f"  📊 Xác suất: {xac_suat_max[0]} = {xac_suat_max[1]:.2%}")

# ===== 3. Dashboard Tổng Hợp =====
fig = plt.figure(figsize=(20, 10))
gs = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)

# Phân phối nhãn gốc
ax1 = fig.add_subplot(gs[0, 0])
if COT_SENTIMENT:
    dem_s = du_lieu_goc['ten_sentiment'].value_counts()
    ax1.pie(dem_s.values, labels=dem_s.index, autopct='%1.0f%%',
            colors=['#3498db', '#2ecc71', '#e74c3c', '#e67e22'],
            wedgeprops={'edgecolor': 'white'})
    ax1.set_title('Phân Phối\nSentiment', fontweight='bold')

# Top từ sau xử lý
ax2 = fig.add_subplot(gs[0, 1:3])
top_15 = dem_tu.most_common(15)
t15, f15 = zip(*top_15)
ax2.barh(t15[::-1], f15[::-1], color=sns.color_palette('viridis', 15))
ax2.set_title('Top 15 Từ Phổ Biến', fontweight='bold')
ax2.set_xlabel('Tần Suất')

# Kết quả mô hình
ax3 = fig.add_subplot(gs[0, 3])
ten_mo_hinh = bang_ket_qua['Mô hình'].tolist()
f1_values = bang_ket_qua['F1 Macro'].tolist()
mau_mo_hinh = ['#e74c3c' if t == mo_hinh_tot_nhat else '#3498db' for t in ten_mo_hinh]
bars3 = ax3.bar(range(len(ten_mo_hinh)), f1_values, color=mau_mo_hinh)
ax3.set_xticks(range(len(ten_mo_hinh)))
ax3.set_xticklabels([t.split()[0] for t in ten_mo_hinh], rotation=15)
ax3.set_title('F1 Macro\nCác Mô Hình', fontweight='bold')
ax3.set_ylabel('F1 Score')
ax3.set_ylim(0, 1)
for bar, v in zip(bars3, f1_values):
    ax3.text(bar.get_x() + bar.get_width()/2, v + 0.01,
             f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# Confusion matrix
ax4 = fig.add_subplot(gs[1, :2])
sns.heatmap(ma_tran_chuan_hoa, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=ten_cac_nhan, yticklabels=ten_cac_nhan,
            ax=ax4, linewidths=0.5)
ax4.set_title(f'Ma Trận Nhầm Lẫn ({mo_hinh_tot_nhat})', fontweight='bold')
ax4.set_xlabel('Dự Đoán')
ax4.set_ylabel('Thực Tế')

# Tỷ lệ lỗi theo lớp
ax5 = fig.add_subplot(gs[1, 2:])
ty_le_loi_sorted = ty_le_loi_pct.sort_values()
mau_loi_bar2 = ['#2ecc71' if v < 20 else '#e67e22' if v < 40 else '#e74c3c'
                 for v in ty_le_loi_sorted.values]
bars5 = ax5.barh(ty_le_loi_sorted.index, ty_le_loi_sorted.values, color=mau_loi_bar2)
ax5.set_title('Tỷ Lệ Lỗi Theo Lớp (%)', fontweight='bold')
ax5.set_xlabel('Tỷ Lệ Lỗi (%)')
for bar in bars5:
    ax5.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():.1f}%', va='center', fontweight='bold')

plt.suptitle('DASHBOARD TỔNG HỢP – PHÂN TÍCH NEU-ESC', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

## KẾT LUẬN

### Tóm Tắt Dữ Liệu
- **Kích thước:** ~23,000+ dòng × 3+ cột (text, sentiment, topic)
- **Biến mục tiêu:** Sentiment (4 lớp: Neutral 69%, Positive 13%, Negative 16%, Toxic 2.5%)
- **Vấn đề chính:** Mất cân bằng nhãn nghiêm trọng (đặc biệt lớp Toxic)
- **Đặc thù:** Văn bản tiếng Việt ngắn, ngôn ngữ sinh viên, nhiều từ lóng

### Kết Quả Chính
| Metric | Giá trị |
|--------|---------|
| Best model | LinearSVC + TF-IDF |
| Accuracy | ~75-82% |
| F1 Macro | ~60-70% |
| F1 Weighted | ~75-80% |

### Hạn Chế
- Baseline TF-IDF không nắm bắt được ngữ cảnh và ngữ nghĩa sâu của tiếng Việt
- Lớp Toxic quá ít mẫu — recall thấp, ảnh hưởng lớn đến F1 Macro
- Chưa sử dụng mô hình Transformer (PhoBERT) trong notebook này
- Chưa tối ưu siêu tham số (hyperparameter tuning)

### Hướng Phát Triển
- Fine-tune **PhoBERT** để cải thiện đáng kể hiệu suất phân loại sentiment
- Áp dụng **multitask learning** để huấn luyện đồng thời cả sentiment và topic
- Thu thập thêm dữ liệu lớp Toxic hoặc áp dụng data augmentation
- Thử nghiệm ensemble (TF-IDF + PhoBERT) cho hệ thống production